# Inspect the first-straight-section merge

This notebook is a **read-only diagnostic** for the `ip == 0` path in `pipeline.inflection_points.inflection_points_curve`. It reproduces the preprocessing used by Step 2 for a real triggering reach, traces second-pass `arcVals` calls only in memory, and draws the retained and temporary geometries.

Default case: African shard 00, `combined_reach_id = 2186`. This reach enters the first-straight branch and is useful for verifying that the first retained bend begins at the reach start after the fix.

In [ ]:
from pathlib import Path
import importlib
import sys
import warnings

import geopandas as gpd
from IPython.display import display
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import trim_mean
from shapely.geometry import Point

# Jupyter may start outside the repository; make the local pipeline package importable.
project_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents, Path('/Users/6256481/Code/river-confinement')]
REPO_ROOT = next((path for path in project_candidates if (path / 'pipeline').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not find the repository. Set REPO_ROOT to the folder containing pipeline/.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline.connect_geometries import merge_centerlines
import pipeline.inflection_points as inflection_module
inflection_module = importlib.reload(inflection_module)
from pipeline.smoothing import SG_smoothing
from pipeline.support import adjust_new_segments, node_position

warnings.filterwarnings('ignore', category=RuntimeWarning)

# Change these only when investigating another real reach.
DATA_ROOT = Path('/Volumes/PhD/confinement')
SWORD_ROOT = Path('/Volumes/PhD/SWORD/v17/GPKG')
CONTINENT = 'af'
SHARD = '00'
COMBINED_REACH_ID = 2186

VECTOR_FILE = DATA_ROOT / 'results' / 'new_segments' / 'vector' / f'{CONTINENT}_{SHARD}_reach_new_segments.gpkg'
NODE_FILE = SWORD_ROOT / f'{CONTINENT}_sword_nodes_v17.gpkg'
SMOOTHING_FILE = DATA_ROOT / 'results' / 'smoothingFactor.csv'

for required in (VECTOR_FILE, NODE_FILE, SMOOTHING_FILE):
    if not required.exists():
        raise FileNotFoundError(f'Missing input: {required}')

print(f'Using {VECTOR_FILE.name}; combined reach {COMBINED_REACH_ID}')
print(f'Inflection-point source: {Path(inflection_module.__file__).resolve()}')

In [ ]:
# Reproduce the relevant Step 2 preparation, without writing any files.
df = adjust_new_segments(gpd.read_file(VECTOR_FILE))
df_reach_wgs84 = df.loc[(df['include_flag'] == '0') & (df['combined_reach_id'] == COMBINED_REACH_ID)].copy()
if df_reach_wgs84.empty:
    raise ValueError(f'No included rows for combined reach {COMBINED_REACH_ID}')

reach_ids = df_reach_wgs84['reach_id'].astype(int).unique().tolist()
where = 'reach_id IN (' + ','.join(map(str, reach_ids)) + ')'
df_nodes_wgs84 = gpd.read_file(NODE_FILE, where=where)

reach_crs = df_reach_wgs84['localCRS'].value_counts().idxmax()
df_reach = df_reach_wgs84.to_crs(reach_crs)
df_nodes = df_nodes_wgs84.to_crs(reach_crs)
line, _, _ = merge_centerlines(df_reach, df, reach_crs)

factor_width = df_reach['combined_reach_width'].iloc[0]
if df_nodes['width'].std() > factor_width / 2:
    factor_width = trim_mean(df_nodes['width'], 0.05)
if np.isnan(factor_width):
    factor_width = df_nodes['width'].mean()

smoothing_table = pd.read_csv(SMOOTHING_FILE)
smoothing_factor = smoothing_table.loc[
    (smoothing_table['combined_reach_width'] - factor_width).abs().idxmin(),
    'smoothFactor',
]
line = SG_smoothing(line, smoothing_factor * int(factor_width), factor_width, id=COMBINED_REACH_ID)
if len(line.coords) < 3:
    line = line.segmentize(1)
df_nodes = node_position(line, df_nodes)

print(f'Local CRS: {reach_crs}; smoothed reach length: {line.length:,.1f} m; width used: {factor_width:,.1f} m')

In [ ]:
# Trace only second-pass calls (return_apex_points=True). The source module is restored immediately.
original_arc_vals = inflection_module.arcVals
second_pass_calls = []

def traced_arc_vals(p1, p2, river_line, df_r, df_n, return_apex_points):
    values = original_arc_vals(p1, p2, river_line, df_r, df_n, return_apex_points)
    if return_apex_points:
        second_pass_calls.append({
            'p1': Point(p1),
            'p2': Point(p2),
            'p1_position_m': river_line.project(Point(p1)),
            'p2_position_m': river_line.project(Point(p2)),
            'segment_sign': values[6],
            'amplitude_m': values[4],
            'bend_width_m': values[0],
            'bend_line': values[2],
        })
    return values

inflection_module.arcVals = traced_arc_vals
try:
    result = inflection_module.inflection_points_curve(line, df_reach, df_nodes)
finally:
    inflection_module.arcVals = original_arc_vals

trace = pd.DataFrame(second_pass_calls).drop(columns=['p1', 'p2', 'bend_line'])
display(trace.round(2))

first_call = second_pass_calls[0]
if not (np.isclose(first_call['p1_position_m'], 0) and first_call['segment_sign'] == 0):
    raise RuntimeError('This reach did not enter the expected first-straight branch; choose another triggering reach.')

returned_inflections = result[2]
returned_bends = result[8]
returned_start = Point(returned_bends[0].coords[0])
returned_start_position = line.project(returned_start)
print(f'First returned bend starts {returned_start_position:,.1f} m from the reach start.')

In [ ]:
def add_line(fig, geom, *, name, color, width=2, dash=None, opacity=1.0):
    xy = np.asarray(geom.coords)
    fig.add_trace(go.Scatter(
        x=xy[:, 0], y=xy[:, 1], mode='lines', name=name,
        line=dict(color=color, width=width, dash=dash), opacity=opacity,
        hovertemplate=f'{name}<extra></extra>',
    ))

fig = go.Figure()
add_line(fig, line, name='Full smoothed reach', color='#9ca3af', width=3)
add_line(fig, first_call['bend_line'], name='Initial straight-classified candidate', color='#f59e0b', width=5, dash='dash')

for index, bend in enumerate(returned_bends):
    add_line(fig, bend, name=f'Returned bend {index}', color='#2563eb' if index == 0 else '#16a34a', width=4)

# Before the fix, the discarded wraparound recomputation has p1 downstream of p2.
wrapped_call = next((call for call in second_pass_calls[1:] if call['p1_position_m'] > call['p2_position_m']), None)
boundary_color = '#dc2626' if returned_start_position > 0.1 else '#16a34a'
boundary_label = (
    f'Unexpected shifted first-bend start ({returned_start_position:,.1f} m)'
    if returned_start_position > 0.1
    else 'First retained bend starts at reach start (corrected behavior)'
)
points = [
    (Point(line.coords[0]), 'Reach start', '#111827', 'circle'),
    (first_call['p2'], 'End of first straight-classified candidate', '#f59e0b', 'diamond'),
    (returned_start, boundary_label, boundary_color, 'x'),
    (Point(line.coords[-1]), 'Reach end', '#111827', 'square'),
]
if wrapped_call is not None:
    points.append((wrapped_call['p1'], 'Wrapped p1: infCoords[-1] (temporary only)', '#9333ea', 'triangle-up'))

for point, label, color, symbol in points:
    fig.add_trace(go.Scatter(
        x=[point.x], y=[point.y], mode='markers+text', name=label,
        text=[label], textposition='top center',
        marker=dict(size=12, color=color, symbol=symbol),
        hovertemplate=f'{label}<br>x=%{{x:.1f}}<br>y=%{{y:.1f}}<extra></extra>',
    ))

fig.update_layout(
    title=f'First-straight merge diagnostic — combined reach {COMBINED_REACH_ID}',
    template='plotly_white', width=1100, height=760,
    legend=dict(orientation='h', yanchor='bottom', y=-0.28, xanchor='left', x=0),
    margin=dict(l=30, r=30, t=60, b=150),
)
fig.update_xaxes(title='Local projected x (m)', scaleanchor='y', scaleratio=1)
fig.update_yaxes(title='Local projected y (m)')
fig.show()

# A focused view makes the start-of-reach boundary shift legible even for a long reach.
focus_length = min(5_000, line.length)
focus_points = np.asarray([line.interpolate(distance).coords[0] for distance in np.linspace(0, focus_length, 250)])
padding = 0.08 * max(np.ptp(focus_points[:, 0]), np.ptp(focus_points[:, 1]))
focus = go.Figure(fig)
focus.update_layout(title=f'Zoom: first {focus_length:,.0f} m of combined reach {COMBINED_REACH_ID}')
focus.update_xaxes(range=[focus_points[:, 0].min() - padding, focus_points[:, 0].max() + padding])
focus.update_yaxes(range=[focus_points[:, 1].min() - padding, focus_points[:, 1].max() + padding])
focus.show()

if wrapped_call is None and returned_start_position <= 0.1:
    print('Corrected behavior verified: no wrapped temporary call was made and the first retained bend begins at the reach start.')
else:
    print('Unexpected behavior: inspect the purple and red markers; a shifted first-bend boundary still leaves an uncovered upstream segment.')